In [0]:
-- Defensive stats for a team's defensemen. Change the settings in params only.
-- Team and position come from each game's roster, so a traded player is only
-- credited for the games he dressed for this team, as a defenseman.
WITH params AS (
  SELECT
    'MTL'      AS team,        -- team abbreviation
    2          AS game_type,   -- 2 = regular season, 3 = playoffs
    20252026   AS season,      -- CAST(NULL AS INT) for all seasons
    10         AS min_games    -- hide players with fewer games
),
games AS (                     -- every game a defenseman dressed for the team
  SELECT r.game_id, r.player_id
  FROM nhl.silver.game_rosters r
  JOIN nhl.gold.dim_game g ON r.game_id = g.game_id
  JOIN nhl.gold.dim_team t ON r.team_id = t.team_id
  CROSS JOIN params pr
  WHERE r.position_code = 'D'
    AND t.abbrev = pr.team
    AND g.game_type = pr.game_type
    AND (pr.season IS NULL OR g.season = pr.season)
),
gp AS (
  SELECT player_id, COUNT(*) AS games_played
  FROM games
  GROUP BY player_id
),
ev AS (                        -- plays in those games where the player had a defensive role
  SELECT gm.player_id AS pid, f.*
  FROM nhl.gold.fact_event f
  JOIN games gm
    ON f.game_id = gm.game_id
   AND gm.player_id IN (f.blocking_player_id, f.hitting_player_id, f.hittee_player_id,
                        f.player_id, f.committed_by_player_id, f.drawn_by_player_id)
  WHERE f.period_type <> 'SO'
    AND f.type_desc_key IN ('blocked-shot', 'hit', 'takeaway', 'giveaway', 'penalty')
),
stats AS (
  SELECT
    pid AS player_id,
    COUNT_IF(type_desc_key = 'blocked-shot' AND blocking_player_id     = pid)  AS blocked_shots,
    COUNT_IF(type_desc_key = 'hit'          AND hitting_player_id      = pid)  AS hits,
    COUNT_IF(type_desc_key = 'hit'          AND hittee_player_id       = pid)  AS hits_taken,
    COUNT_IF(type_desc_key = 'takeaway'     AND player_id              = pid)  AS takeaways,
    COUNT_IF(type_desc_key = 'giveaway'     AND player_id              = pid)  AS giveaways,
    SUM(IF(type_desc_key = 'penalty' AND committed_by_player_id = pid,
           COALESCE(penalty_duration, 0), 0))                                  AS pim,
    COUNT_IF(type_desc_key = 'penalty'      AND drawn_by_player_id     = pid)  AS penalties_drawn
  FROM ev
  GROUP BY pid
)
SELECT
  p.full_name,
  gp.player_id,
  gp.games_played,
  COALESCE(s.blocked_shots, 0)                                    AS blocked_shots,
  ROUND(COALESCE(s.blocked_shots, 0) / gp.games_played, 2)        AS blocks_per_game,
  COALESCE(s.hits, 0)                                             AS hits,
  ROUND(COALESCE(s.hits, 0) / gp.games_played, 2)                 AS hits_per_game,
  COALESCE(s.hits_taken, 0)                                       AS hits_taken,
  COALESCE(s.takeaways, 0)                                        AS takeaways,
  COALESCE(s.giveaways, 0)                                        AS giveaways,
  COALESCE(s.takeaways, 0) - COALESCE(s.giveaways, 0)             AS takeaway_giveaway_diff,
  COALESCE(s.pim, 0)                                              AS pim,
  COALESCE(s.penalties_drawn, 0)                                  AS penalties_drawn
FROM gp
JOIN nhl.gold.dim_player p ON gp.player_id = p.player_id
LEFT JOIN stats s          ON s.player_id  = gp.player_id
WHERE gp.games_played >= (SELECT min_games FROM params)
ORDER BY gp.games_played DESC, blocked_shots DESC;